# TPCDrum 단계별 재구성

기존 CNN용 커널에서 실행합니다. 새 DDE digi와 물리 보정은 아직 확인 전입니다.
기본은 입력 검사(`inspect`)입니다. `all`은 단계별 ROOT를 저장하며, 보정이 없으면 실패 이유를 남깁니다.
질량·120° 정확도를 보장하는 완성 모델이 아닙니다. 설명은 같은 폴더의 `README.txt`를 참고하세요.


In [ ]:
from pathlib import Path
import json,sys,shutil
from datetime import datetime
from uuid import uuid4

PACKAGE=Path.cwd().resolve()  # 이 노트북을 reconstruction 폴더에서 열기
if not (PACKAGE/'run.py').is_file():
    raise FileNotFoundError('PACKAGE를 tpcdrum/reconstruction 절대경로로 지정하세요.')
INPUT_ROOT=PACKAGE.parent/'simulation/C12_DDE_allcomb_3C3_digi_SiEdep.root'
STAGE='inspect'  # inspect, all, finding, fitting, vertex, association, energy, kinematics 등
ENTRIES=[0,5]  # 작은 점검부터. 전체는 None
REPRESENTATION='direct_waveform'  # 또는 space_points
FINDING_METHOD='hough'  # cnn, cnn_hough, ransac, tls, waveform_hypotheses
FINDING_CHECKPOINT=None  # CNN일 때 실제로 훈련한 checkpoint 절대경로
PREVIOUS_ROOT=None  # 개별 단계 실행 시 이전 단계 .root

cfg=json.loads((PACKAGE/'config.json').read_text())
cfg.update(input_roots=[str(INPUT_ROOT.resolve())],entries=ENTRIES,representation=REPRESENTATION,
           root_executable=shutil.which('root'))
cfg['finding'].update(method=FINDING_METHOD,checkpoint=FINDING_CHECKPOINT)
cfg['navigator'].update(repo=str(PACKAGE.parents[1]),
    parameters=str(PACKAGE.parent/'simulation/SimParameters.mac'))
# 보정값을 얻은 뒤 cfg['drift'], cfg['si_geometry'], cfg['energy']를 여기에 명시합니다.
# nominal 650 Torr 또는 TB 값만으로 물리적 drift 보정을 완료했다고 표시하지 마세요.

stamp=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid4().hex[:8]
RUN_DIR=PACKAGE/'results'/stamp
RUN_DIR.mkdir(parents=True,exist_ok=False)
CONFIG=RUN_DIR/'run_config.json'
CONFIG.write_text(json.dumps(cfg,indent=2))
OUTPUT=RUN_DIR/(STAGE+'.root')
print('Python:',sys.executable,'설정:',CONFIG,'결과:',RUN_DIR,sep='\n')


In [ ]:
import subprocess,time

command=[sys.executable,'-u',str(PACKAGE/'run.py'),'--stage',STAGE,'--config',str(CONFIG)]
if STAGE=='all':command+=['--output-dir',str(RUN_DIR/'stages')]
elif STAGE not in ('inspect',):
    command+=['--output',str(OUTPUT)]
    if STAGE not in ('ingest','truth'):
        if PREVIOUS_ROOT is None:raise ValueError('PREVIOUS_ROOT가 필요합니다.')
        command+=['--input',str(Path(PREVIOUS_ROOT).expanduser().resolve(strict=True))]
LOG=RUN_DIR/'run.log'
with LOG.open('x') as stream:
    proc=subprocess.Popen(command,stdout=stream,stderr=subprocess.STDOUT)
    offset=0;last_heartbeat=time.monotonic()
    try:
        while proc.poll() is None:
            with LOG.open() as reader:
                reader.seek(offset);new=reader.read();offset=reader.tell()
            if new:print(new,end='',flush=True);last_heartbeat=time.monotonic()
            elif time.monotonic()-last_heartbeat>=30:
                print('실행 중. PID:',proc.pid,'로그:',LOG,flush=True);last_heartbeat=time.monotonic()
            time.sleep(1)
    except BaseException:
        proc.terminate()
        try:proc.wait(timeout=5)
        except subprocess.TimeoutExpired:proc.kill();proc.wait(timeout=5)
        raise
with LOG.open() as reader:
    reader.seek(offset);print(reader.read(),end='')
print('종료 코드:',proc.returncode,'로그:',LOG)
if proc.returncode==4:
    print('ROOT 진단 파일은 저장됐지만 물리 결과가 완성되지 않았습니다. missing/ambiguous 이유를 확인하세요.')
elif proc.returncode:
    raise RuntimeError('로그의 첫 오류를 확인하세요.')


## 저장된 ROOT 확인

아래 셀은 새 ROOT의 요약과 첫 이벤트 진단을 읽습니다. ROOT 없이도 표를 볼 수 있습니다.
큰 파형·궤적 그림은 `simulation/C12_view_reconstruction.C`에서 보거나 PDF로 저장할 수 있습니다.


In [ ]:
import uproot
from IPython.display import display

ROOT_FILES=sorted(RUN_DIR.rglob('*.root'))
for path in ROOT_FILES:
    with uproot.open(path) as source:
        print(path)
        if 'metrics' in source:
            m=source['metrics'].arrays(library='np')
            display(dict(zip(m['name'].tolist(),m['value'].tolist())))
        if 'events' in source:
            row=source['events']['record_json'].array(entry_start=0,entry_stop=1,library='np')
            print(json.dumps(json.loads(row[0]),ensure_ascii=False,indent=2)[:5000])
